[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day17-naive-batching-pitfalls.ipynb)
# Day 17 — Naive Batching Pitfalls
**Tag:** CPU-OK (T4 optional) · **Time:** ~45 min

Static batching is the simplest way to batch: collect up to B requests (or wait T ms), left-pad to the longest prompt, one batched `generate()`. Today you quantify its two taxes — **padding waste** and the **straggler effect** — then measure them on a real model.

In [ ]:
!pip install -q transformers torch --index-url https://download.pytorch.org/whl/cpu
# Expected output: (quiet install, no errors)

## 1. Tax #1: padding waste, as a reusable function

Waste = padded tokens that go through every layer and produce nothing. Rule of thumb: waste fraction ≈ 1 − mean_len / max_len.

In [ ]:
def padding_waste(prompt_lens):
    m = max(prompt_lens)
    total = m * len(prompt_lens)
    wasted = total - sum(prompt_lens)
    return wasted, total, wasted / total

# Worked example from the packet: [2000, 100, 100, 100]
w, t, frac = padding_waste([2000, 100, 100, 100])
print(f"wasted={w}, total={t}, waste={frac:.1%}")
assert abs(frac - 0.7125) < 1e-9

# Your turn: a uniform workload (classification-style API)
w2, t2, frac2 = padding_waste([512, 500, 520, 505])
print(f"uniform workload waste={frac2:.1%}")
# Expected output:
# wasted=5700, total=8000, waste=71.2%
# uniform workload waste=1.9%

## 2. Tax #2: the straggler effect, simulated step by step

The batch runs until its *longest* output finishes. Short requests hold GPU slots generating masked-out tokens. Utilization = useful token-steps / (B × max_steps).

In [ ]:
def straggler_sim(out_lens, ms_per_step=4.8):
    B, longest = len(out_lens), max(out_lens)
    useful = sum(min(l, longest) for l in out_lens)  # = sum(out_lens)
    possible = B * longest
    util = useful / possible
    print(f"batch={out_lens} -> {longest} decode steps")
    for i, l in enumerate(out_lens):
        waited_ms = longest * ms_per_step
        needed_ms = l * ms_per_step
        print(f"  req{i}: needed {l} steps ({needed_ms:5.0f} ms), waited {longest} steps ({waited_ms:5.0f} ms) -> {waited_ms/needed_ms:.1f}x slower")
    print(f"utilization = {useful}/{possible} = {util:.1%}")
    return util

u = straggler_sim([50, 50, 50, 400])
assert abs(u - 0.34375) < 1e-9
print()
u2 = straggler_sim([120, 130, 128, 125])  # uniform workload
# Expected output:
# batch=[50, 50, 50, 400] -> 400 decode steps
#   req0: needed 50 steps (  240 ms), waited 400 steps ( 1920 ms) -> 8.0x slower
#   ...
# utilization = 550/1600 = 34.4%
# batch=[120, 130, 128, 125] -> 130 decode steps
# utilization = 503/520 = 96.7%

## 3. Real measurement: sequential vs static batch on SmolLM2-135M (CPU)

Time 4 sequential generations vs one left-padded batch of 4. Short prompts keep CPU runtime sane.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
tok.pad_token = tok.eos_token  # SmolLM2 has no pad token; use EOS for padding
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model.eval()
prompts = ["The capital of France is", "Photosynthesis converts",
           "def fibonacci(n):", "The quick brown fox"]

# Sequential baseline (your Day 16 server)
t0 = time.perf_counter()
seq_ids = []
for p in prompts:
    ids = tok(p, return_tensors="pt").input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=20, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    seq_ids.append(out[0].tolist())
seq_s = time.perf_counter() - t0
seq_tps = 4 * 20 / seq_s
print(f"sequential: {seq_s:.1f}s for 80 tokens -> {seq_tps:.1f} tok/s")

# Static batch: left-pad to longest, one generate() call
enc = tok(prompts, return_tensors="pt", padding="longest", padding_side="left")
t0 = time.perf_counter()
with torch.no_grad():
    bout = model.generate(**enc, max_new_tokens=20, do_sample=False,
                          pad_token_id=tok.eos_token_id)
bat_s = time.perf_counter() - t0
bat_tps = 4 * 20 / bat_s
print(f"static batch B=4: {bat_s:.1f}s for 80 tokens -> {bat_tps:.1f} tok/s")
print(f"speedup: {bat_tps/seq_tps:.2f}x")

# Correctness: batched outputs must match sequential (modulo padding)
for i, p in enumerate(prompts):
    n_prompt = enc.attention_mask[i].sum().item()
    gen = bout[i][n_prompt:].tolist()  # strip prompt via mask, not fixed length
    assert gen == seq_ids[i][-20:], f"mismatch on prompt {i}"
print("OK: batched tokens identical to sequential")
# Expected output (CPU, SmolLM2-135M; your numbers will differ):
# sequential: ~25-40s for 80 tokens -> ~2-3 tok/s
# static batch B=4: ~8-15s -> ~3-4x speedup
# OK: batched tokens identical to sequential

## 4. The collector trade-off: batch size vs timeout

Collector: gather until B requests or T ms elapse. Simulate Poisson arrivals at rate λ; the average fill tells you which knob is binding.

In [ ]:
import random

def collector_sim(lam=5.0, B=4, T_ms=100, sim_s=200, seed=0):
    rng = random.Random(seed)
    t = rng.expovariate(lam)  # first arrival
    fills, waits = [], []
    while t < sim_s:
        t_start, batch = t, 1
        deadline = t_start + T_ms / 1000.0
        while batch < B:
            gap = rng.expovariate(lam)
            if t + gap > deadline:   # timeout fires first
                t = deadline
                break
            t += gap
            batch += 1
        fills.append(batch)
        waits.append(t - t_start)
        t += rng.expovariate(lam)    # next window starts at next arrival
    return sum(fills) / len(fills), sum(waits) / len(waits)

for T in [0, 50, 100, 200, 600]:
    avg_fill, avg_wait = collector_sim(lam=5.0, B=4, T_ms=T)
    print(f"T={T:3d} ms  avg fill={avg_fill:.2f}/4  avg first-arrival wait={avg_wait*1000:5.0f} ms")
# Expected output:
# T=  0 ms  avg fill=1.00/4  avg first-arrival wait=    0 ms
# T= 50 ms  avg fill=1.25/4  avg first-arrival wait=   50 ms
# T=100 ms  avg fill=1.50/4  avg first-arrival wait=  100 ms
# T=200 ms  avg fill=2.00/4  avg first-arrival wait=  200 ms
# T=600 ms  avg fill=3.34/4  avg first-arrival wait=  470 ms   <- near-full batches need ~600 ms TTFT tax

print("\nPacket check: predicted avg fill at T=100ms, lam=5/s:", 1 + 5*0.1)


## Wrap-up

**Measured today:** padding waste on a skewed batch (~71%), straggler utilization (~34%), real sequential-vs-batch speedup on CPU, and the collector curve (full batches need T ≈ (B−1)/λ).

**Checkpoints** (answers in the Day 17 packet PDF):
1. Batch prompts [2000, 100, 100, 100]: padding waste?
2. Same batch, outputs [50, 50, 50, 400]: decode utilization?
3. λ = 5 req/s, B = 4, T = 100 ms: which collector knob binds?

**Tomorrow (Day 18):** Continuous batching (Orca) — schedule per *iteration*, refill finished slots immediately, and watch 34% utilization climb past 90% on the same workload.